In [ ]:
import gc
import itertools
import math
import os
import random
import sys
from collections import Counter, defaultdict
from copy import deepcopy
from dataclasses import dataclass
from functools import partial
from pathlib import Path
from typing import Any, Callable, Literal, TypeAlias

import einops
import numpy as np
import pandas as pd
import torch as t
from datasets import load_dataset
from IPython.display import clear_output, display
from jaxtyping import Float, Int
from rich import print as rprint
from rich.table import Table

from transformer_lens import HookedTransformer, HookedTransformerConfig
from tabulate import tabulate
from torch import Tensor, nn
from torch.nn import functional as F
from tqdm.auto import tqdm
from transformer_lens import ActivationCache, loading_from_pretrained
from transformer_lens.hook_points import HookPoint
from transformer_lens.utils import get_act_name, to_numpy
from transformer_lens import utils

import matplotlib.pyplot as plt
import seaborn as sns
import torch

from scipy.sparse import csr_array
from scipy.sparse.csgraph import maximum_bipartite_matching, min_weight_full_bipartite_matching

device = t.device("mps" if t.backends.mps.is_available() else "cuda" if t.cuda.is_available() else "cpu")

## Model & State_dict Loading 

In [2]:
import importlib

# ------------------- Load Model Config (Function) -------------------

def load_named_config(module_name: str, config_name: str) -> dict:
    """
    Import a module that defines CONFIGS: Dict[str, Dict[str, Any]]
    and return CONFIGS[config_name].
    """
    try:
        mod = importlib.import_module(module_name)
    except Exception as e:
        raise ImportError(f"Could not import config module '{module_name}': {e}") from e

    if not hasattr(mod, "CONFIGS"):
        raise AttributeError(f"Module '{module_name}' does not define CONFIGS.")

    CONFIGS = getattr(mod, "CONFIGS")
    if config_name not in CONFIGS:
        available = ", ".join(sorted(CONFIGS.keys()))
        raise KeyError(f"Config '{config_name}' not found in {module_name}. Available: {available}")

    return dict(CONFIGS[config_name])  # copy so we can tweak


## Prompt Sentences Data

In [3]:
import pickle

prompts_file = "100_prompts"

with open(f"./{prompts_file}.pkl", "rb") as f:
    prompts = pickle.load(f)

print(f"Loaded {len(prompts)} prompts from ./{prompts_file}.pkl")

Loaded 100 prompts from ./100_prompts.pkl


## Cache attention-head activations (all architectures)

Because this step is slow, you can run it **one architecture at a time** by switching the model config.

### Sweep over model configurations

- `layer_list = [2, 4, 8, 12]`
- `head_list  = [8, 12, 16]`
- `attn_list  = [True, False]`
- `SEEDS      = list(range(1, 51))`

In [ ]:
# Paths
SCRATCH = "Path to root directory"
MODELS_DIR = SCRATCH + "chkpts"
ADAM_ATTN_DIR = SCRATCH + "cache_attn"
ADAMW_ATTN_DIR = SCRATCH + "cache_attn_wd"

epoch = 1
shard = 9

# If processing AdamW refits -> Set it to True; else False
WD = True

# Loop over model configurations
layer_list = [2, 4, 8, 12]
head_list = [8, 12, 16]
attn_list = [True, False]
SEEDS = [i for i in range(1,51)]

    
for NUM_LAYERS in layer_list:
    for NUM_HEADS in head_list:
        for attn_only in attn_list:

            if (NUM_LAYERS == 12 and (NUM_HEADS == 16 or NUM_HEADS == 8)) or (NUM_LAYERS == 12 and NUM_HEADS == 12 and attn_only==True) or (NUM_LAYERS != 12 and NUM_HEADS == 12):
                print(f"Skipping l{NUM_LAYERS}_h{NUM_HEADS}_attnonly={attn_only}")
                continue
            else:
                print(f"Processing l{NUM_LAYERS}_h{NUM_HEADS}_attnonly={attn_only}")
            
                chkpts_list = [ "model_0.pt", "model_66285.pt", "model_132570.pt", "model_198855.pt", "model_265140.pt", "final.pt" ]
                chkpt = 5
                chkpt_file = chkpts_list[chkpt]
                print(f"Using checkpoint: {chkpt_file}")

                # ------------------- Load Model Config -------------------
                if (NUM_LAYERS == 12 and NUM_HEADS == 12):
                    arch = "gpt2"
                    SEEDS = [i for i in range(1,6)]
                else:
                    arch = f"l{NUM_LAYERS}_h{NUM_HEADS}_attn" if attn_only else f"l{NUM_LAYERS}_h{NUM_HEADS}"
                    
                if WD:
                    arch = arch + '_wd'
                print(f"arch: ", arch)
                cfg_dict = load_named_config("model_configs", arch)


                # ------------------- Model Configuration -------------------

                # Build HookedTransformerConfig using the loaded config
                cfg = HookedTransformerConfig(
                    n_layers=cfg_dict["n_layers"],
                    d_model=cfg_dict["d_model"],
                    n_heads=cfg_dict["n_heads"],
                    d_head=cfg_dict["d_head"],
                    d_mlp=cfg_dict.get("d_mlp", None),
                    n_ctx=cfg_dict["n_ctx"],
                    act_fn=cfg_dict.get("act_fn", "gelu"),
                    d_vocab=cfg_dict["d_vocab"],
                    init_weights=True,
                    tokenizer_name=cfg_dict["tokenizer_name"],
                    model_name=cfg_dict.get("model_name", arch),
                    attn_only=cfg_dict.get("attn_only", False),
                )


                # Load models
                models = []
                for SEED in SEEDS:
                    cfg.seed = SEED
                    cfg.init_weights = True
                    model = HookedTransformer(cfg)
                    models.append(model)

                for ind, SEED in enumerate(SEEDS):
                    model = models[ind]
                    if (arch == "gpt2") or (arch == "gpt2_wd"):
                        state_dict_path = f"{MODELS_DIR}/{arch}/gpt2_seed{SEED}_shard{shard}_epoch{epoch}_owt/{chkpt_file}"
                    else:
                        if attn_only:
                            state_dict_path = f"{MODELS_DIR}/{arch}/causal_attn_only_l{NUM_LAYERS}_h{NUM_HEADS}_seed{SEED}_epoch{epoch}_c4_gelu/{chkpt_file}"
                        else:
                            state_dict_path = f"{MODELS_DIR}/{arch}/causal_attn_l{NUM_LAYERS}_h{NUM_HEADS}_seed{SEED}_epoch{epoch}_c4_gelu/{chkpt_file}"
                    model_state_dict = t.load(state_dict_path)
                    model.load_and_process_state_dict(model_state_dict, fold_ln=False)

                # Setting device to CPU as GPU memory is insufficient for this computation, but for smaller number of prompts/models it can be set to GPU
                device = 'cpu'

                # run prompts to collect caches (using CPU to avoid CUDA OOM)
                prompts_cache = []
                for prompt in prompts:
                    cache_for_prompt = []
                    for ind in range(len(SEEDS)):
                        _, cache_i = models[ind].run_with_cache(prompt, remove_batch_dim=True)
                        # Keep cache on CPU
                        cache_i = cache_i.to('cpu')
                        cache_for_prompt.append(cache_i)
                    prompts_cache.append(cache_for_prompt)


                # Free memory held by models
                del models
                del model_state_dict
                torch.cuda.empty_cache()

                num_prompts = len(prompts_cache)
                num_seeds = len(prompts_cache[0])
                num_layers = NUM_LAYERS

                attn_activations = []

                for prompt_idx in range(num_prompts):
                    prompt_acts = []
                    for seed_idx in range(num_seeds):
                        seed_acts = []
                        cache = prompts_cache[prompt_idx][seed_idx]
                        for layer in range(num_layers):
                            key = utils.get_act_name('z', layer, 'attn')
                            # shape: [seq_len, d_mlp]
                            #print(key, cache)
                            act = cache[key].detach().cpu()
                            seed_acts.append(act)
                        prompt_acts.append(seed_acts)
                    attn_activations.append(prompt_acts)

                if attn_only:
                    if WD:
                        f_name = f"{ADAMW_ATTN_DIR}/l{NUM_LAYERS}_h{NUM_HEADS}_s{len(prompts)}_i{len(SEEDS)}_attnonly" 
                    else:
                        f_name = f"{ADAM_ATTN_DIR}/l{NUM_LAYERS}_h{NUM_HEADS}_s{len(prompts)}_i{len(SEEDS)}_attnonly"     
                else:
                    if WD:
                        f_name = f"{ADAMW_ATTN_DIR}/l{NUM_LAYERS}_h{NUM_HEADS}_s{len(prompts)}_i{len(SEEDS)}"
                    else:
                        f_name = f"{ADAM_ATTN_DIR}/l{NUM_LAYERS}_h{NUM_HEADS}_s{len(prompts)}_i{len(SEEDS)}"
                os.makedirs(os.path.dirname(f_name), exist_ok=True)
                with open(f_name, "wb") as f:
                    print(f"Saving attention activations to {f_name}")
                    pickle.dump(attn_activations, f)


## Loading cached activation

In [ ]:
import re

# Number of instances in the cache files
i = 50
# Number of prompts in the cache files
s = 100

WD = True
cache_dir = ADAM_ATTN_DIR + "/"
if WD:
    cache_dir = ADAMW_ATTN_DIR + "/"

cache_files = [
    f"l2_h8_s{s}_i{i}",
    f"l2_h8_s{s}_i{i}_attnonly",
    f"l2_h16_s{s}_i{i}",
    f"l2_h16_s{s}_i{i}_attnonly",
    f"l4_h8_s{s}_i{i}",
    f"l4_h8_s{s}_i{i}_attnonly",
    f"l4_h16_s{s}_i{i}",
    f"l4_h16_s{s}_i{i}_attnonly",
    f"l8_h8_s{s}_i{i}",
    f"l8_h8_s{s}_i{i}_attnonly",
    f"l8_h16_s{s}_i{i}",
    f"l8_h16_s{s}_i{i}_attnonly",
    f"l12_h12_s{s}_i5",
]

cfg_cache = []
cfg_list = []

for file_i, file in enumerate(cache_files):
    cached_file = cache_dir + file
    with open(cached_file, "rb") as fp:
        prompts_cache = pickle.load(fp)
    cfg_cache.append(prompts_cache)

    # Extract config values from filename
    match = re.search(r"l(\d+)_h(\d+)_s(\d+)_i(\d+)", file)
    if match:
        num_layers = int(match.group(1))
        num_heads = int(match.group(2))
        num_prompts = int(match.group(3))
        num_models = int(match.group(4))
    else:
        num_layers = num_heads = num_prompts = num_models = None

    cfg_i = {}
    cfg_i['NUM_LAYERS'] = num_layers
    cfg_i['NUM_HEADS'] = num_heads
    cfg_i['NUM_PROMPTS'] = num_prompts
    cfg_i['NUM_MODELS'] = num_models
    cfg_i['cache_file'] = file
    cfg_i['ATTN_ONLY'] = "_attnonly" in file

    cfg_list.append(cfg_i)


In [6]:
cfg_list

[{'NUM_LAYERS': 2,
  'NUM_HEADS': 8,
  'NUM_PROMPTS': 100,
  'NUM_MODELS': 50,
  'cache_file': 'l2_h8_s100_i50',
  'ATTN_ONLY': False},
 {'NUM_LAYERS': 2,
  'NUM_HEADS': 8,
  'NUM_PROMPTS': 100,
  'NUM_MODELS': 50,
  'cache_file': 'l2_h8_s100_i50_attnonly',
  'ATTN_ONLY': True},
 {'NUM_LAYERS': 2,
  'NUM_HEADS': 16,
  'NUM_PROMPTS': 100,
  'NUM_MODELS': 50,
  'cache_file': 'l2_h16_s100_i50',
  'ATTN_ONLY': False},
 {'NUM_LAYERS': 2,
  'NUM_HEADS': 16,
  'NUM_PROMPTS': 100,
  'NUM_MODELS': 50,
  'cache_file': 'l2_h16_s100_i50_attnonly',
  'ATTN_ONLY': True},
 {'NUM_LAYERS': 4,
  'NUM_HEADS': 8,
  'NUM_PROMPTS': 100,
  'NUM_MODELS': 50,
  'cache_file': 'l4_h8_s100_i50',
  'ATTN_ONLY': False},
 {'NUM_LAYERS': 4,
  'NUM_HEADS': 8,
  'NUM_PROMPTS': 100,
  'NUM_MODELS': 50,
  'cache_file': 'l4_h8_s100_i50_attnonly',
  'ATTN_ONLY': True},
 {'NUM_LAYERS': 4,
  'NUM_HEADS': 16,
  'NUM_PROMPTS': 100,
  'NUM_MODELS': 50,
  'cache_file': 'l4_h16_s100_i50',
  'ATTN_ONLY': False},
 {'NUM_LAYERS': 4,

## Creating "Distance matrix" for meta-SNE using L2 distance.
#### To cater for different dimension arising due varying prompt-tokens length, we average the activations over the dimenion decided by prompt-tokens length. 

In [ ]:
SCRATCH = "Path to root directory"
msne_df_dir = "msne_attn"

WD = False
if WD:
    msne_df_dir = "msne_attn_wd"

for cfg, p_cache in enumerate(cfg_cache):
    NUM_LAYERS = cfg_list[cfg]['NUM_LAYERS']
    NUM_MODELS = cfg_list[cfg]['NUM_MODELS']
    NUM_HEADS = cfg_list[cfg]['NUM_HEADS']
    NUM_PROMPTS = cfg_list[cfg]['NUM_PROMPTS']
    ATTN_ONLY = cfg_list[cfg]['ATTN_ONLY']
    CACHE_FILE = cfg_list[cfg]['cache_file']

    if NUM_LAYERS == 8 and NUM_HEADS == 16 and ATTN_ONLY==True:

        rows = []

        for layer in range(NUM_LAYERS):
            for model in range(NUM_MODELS):
                for i in range(NUM_PROMPTS):
                    di = p_cache[i][model][layer]
                    
                    for j in range(NUM_PROMPTS):
                        dj = p_cache[j][model][layer]

                        for head in range(NUM_HEADS):

                            di_head_mean = t.mean(di[:,head,:], 0)
                            dj_head_mean = t.mean(dj[:,head,:], 0)

                            dist = t.norm(di_head_mean - dj_head_mean, p=2).item()
                            rows.append({
                                'layer': layer,
                                'model': model,
                                'attn_only': ATTN_ONLY,
                                'head': head,
                                'i': i,
                                'j': j,
                                'dist': dist
                            })

        df = pd.DataFrame(rows)
        df_name = SCRATCH + f"df/{msne_df_dir}/df_avg_{CACHE_FILE}"
        df.to_pickle(df_name)


## Concatenating distance matrices belonging to all architectures, which can be finally fed to t-SNE.

In [ ]:
SCRATCH = "Path to root directory"
NUM_CFG = len(cfg_cache)
NUM_PROMPTS = 100
dist_cols = [f"d_{i}" for i in range(NUM_PROMPTS*NUM_PROMPTS)]


msne_df_dir = "msne_attn"
WD = True
if WD:
    msne_df_dir = "msne_attn_wd"

df_list = []

for cfg, p_cache in enumerate(cfg_cache):
    CACHE_FILE = cfg_list[cfg]['cache_file']
    df_file = pd.read_pickle(SCRATCH + f"df/{msne_df_dir}/df_avg_{CACHE_FILE}")

    # Adjust indices as needed
    df_file['cfg'] = cfg + 1
    df_file['layer'] = df_file['layer'] + 1
    df_file['head'] = df_file['head'] + 1
    df_file['model'] = df_file['model'] + 1

    # Group and expand dist_list columns
    df_file = df_file.groupby(['cfg', 'layer', 'head', 'model', 'attn_only'])['dist'].apply(list).reset_index(name='dist_list')


    df_file[dist_cols] = pd.DataFrame(df_file.dist_list.tolist(), index=df_file.index)

    df_list.append(df_file)

# Concatenate all at once for efficiency
df = pd.concat(df_list, ignore_index=True)

In [ ]:
df

,cfg,layer,head,model,attn_only,dist_list,d_0,d_1,d_2,d_3,...,d_9990,d_9991,d_9992,d_9993,d_9994,d_9995,d_9996,d_9997,d_9998,d_9999
0,1,1,1,1,False,"[0.0, 0.16544495522975922, 0.17434245347976685...",0.0,0.165445,0.174342,0.168214,...,0.154512,0.173335,0.193678,0.171794,0.140887,0.161780,0.190793,0.135135,0.156748,0.0
1,1,1,1,2,False,"[0.0, 0.39866501092910767, 0.4556886851787567,...",0.0,0.398665,0.455689,0.344776,...,0.350283,0.326309,0.369628,0.413433,0.384941,0.385823,0.360982,0.409601,0.353478,0.0
2,1,1,1,3,False,"[0.0, 0.33857592940330505, 0.3486616909503937,...",0.0,0.338576,0.348662,0.293755,...,0.321773,0.295208,0.390369,0.384451,0.340050,0.358237,0.375578,0.350683,0.315780,0.0
3,1,1,1,4,False,"[0.0, 0.2857029438018799, 0.2330782264471054, ...",0.0,0.285703,0.233078,0.298146,...,0.359370,0.352599,0.359662,0.381926,0.344186,0.348718,0.384815,0.265458,0.263691,0.0
4,1,1,1,5,False,"[0.0, 0.2192823886871338, 0.23467083275318146,...",0.0,0.219282,0.234671,0.214365,...,0.215106,0.209504,0.228392,0.211809,0.208161,0.206425,0.217366,0.221655,0.191029,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
34315,13,12,12,1,False,"[0.0, 6.885926246643066, 7.799212455749512, 6....",0.0,6.885926,7.799212,6.757257,...,8.628075,7.600427,7.773159,7.749729,8.014085,8.514429,8.754876,7.411825,6.975343,0.0
34316,13,12,12,2,False,"[0.0, 2.5815422534942627, 2.9339139461517334, ...",0.0,2.581542,2.933914,2.584414,...,4.647665,3.205686,2.906845,3.012243,3.485058,4.312740,2.880618,3.652880,3.411362,0.0
34317,13,12,12,3,False,"[0.0, 1.6944726705551147, 1.5343199968338013, ...",0.0,1.694473,1.534320,1.785667,...,3.857955,2.144024,2.088552,3.036145,2.461594,2.509736,2.344680,2.630498,2.251204,0.0
34318,13,12,12,4,False,"[0.0, 3.5214202404022217, 3.664677143096924, 2...",0.0,3.521420,3.664677,2.816428,...,7.167874,7.086103,9.143344,6.350374,6.912832,6.345163,5.764539,5.143262,5.623318,0.0


## Utils for t-SNE & plots

In [6]:
df = df.rename(columns={'cfg': 'arch'})

dist_cols = [f"d_{i}" for i in range(10000)]

col_list = ['arch', 'layer', 'model', 'attn_only']

for col in col_list:
    df[col] = df[col].astype(str)

In [ ]:
mapping = {
    str(i+1): f"Layers={cfg['NUM_LAYERS']}, Heads={cfg['NUM_HEADS']}, {'Attn-only' if cfg['ATTN_ONLY'] else 'MLP'}"
    for i, cfg in enumerate(cfg_list)
}

df['arch'] = df['arch'].map(mapping)


In [8]:
import re
import numpy as np

def extract_num_layers(arch_str):
    if pd.isna(arch_str):
        return None
    m = re.search(r'(?i)layers?\s*=?\s*(\d+)', arch_str)
    return int(m.group(1)) if m else None

def extract_num_heads(arch_str):
    if pd.isna(arch_str):
        return None
    m = re.search(r'(?i)heads?\s*=?\s*(\d+)', arch_str)
    return int(m.group(1)) if m else None

def extract_attn_only_flag(arch_str):
    if pd.isna(arch_str):
        return None
    s = arch_str.lower()
    if 'attn' in s:
        return True
    if 'mlp' in s:
        return False
    return None

# parse columns
df['num_layers'] = df['arch'].apply(extract_num_layers)
df['num_heads'] = df['arch'].apply(extract_num_heads)
df['attn_only_parsed'] = df['arch'].apply(extract_attn_only_flag)

# compute ratio safely (layer may be string currently)
def safe_ratio(row):
    try:
        layer_idx = int(row['layer'])
        nl = row['num_layers']
        if nl is None or nl == 0 or np.isnan(nl):
            return np.nan
        return layer_idx / nl
    except Exception:
        return np.nan

df['l_ratio'] = df.apply(safe_ratio, axis=1)

# keep num_heads as string for plotting/mapping (preserve missing as 'None')
df['num_heads'] = df['num_heads'].astype(object).where(df['num_heads'].notna(), None)
df['num_heads'] = df['num_heads'].astype(str)

## meta-SNE (t-SNE)

In [9]:
from sklearn.manifold import TSNE
import plotly.express as px


features = df[dist_cols].values

tsne = TSNE(n_components=2, random_state=42, perplexity=10000.0)
projections = tsne.fit_transform(features)

In [ ]:
# Legend mapping for arch
arch_legend_map = {
    str(i+1): f"Layers={cfg['NUM_LAYERS']}, Heads={cfg['NUM_HEADS']}, {'Attn-only' if cfg['ATTN_ONLY'] else 'MLP'}"
    for i, cfg in enumerate(cfg_list)
}

arch_main = [arch_legend_map[str(i)] for i in range(1, 13, 2)]
arch_light = [arch_legend_map[str(i)] for i in range(2, 13, 2)]

unique_arch = df['arch'].astype(str).unique()
arch_group_map = {arch: ('main' if arch in arch_main else 'light') for arch in unique_arch}

base_colors = px.colors.qualitative.Plotly[:5] + [px.colors.qualitative.Dark24[15]]
arch_color_map = {arch: base_colors[i] for i, arch in enumerate(arch_main)}

import plotly.colors as pc
def lighten_color(color, factor=0.5):
    rgb = pc.hex_to_rgb(color)
    light_rgb = tuple(int(255 - (255 - c) * factor) for c in rgb)
    return pc.label_rgb(light_rgb)

arch_color_full_map = {}
for arch in unique_arch:
    if arch in arch_main:
        color = arch_color_map[arch]
    else:
        # Find corresponding main arch by replacing 'Attn_only' with 'MLP' or vice versa
        if 'Attn_only' in arch:
            main_arch = arch.replace('Attn_only', 'MLP')
        else:
            main_arch = arch.replace('MLP', 'Attn_only')
        color = lighten_color(arch_color_map.get(main_arch, base_colors[0]))
    arch_color_full_map[arch] = color

df['arch_label'] = df['arch']

#per = 2500
per = 1500
fig = px.scatter(
    x=projections[:,0], y=projections[:,1],
    color=df['arch_label'],
    color_discrete_map=arch_color_full_map,
    labels={'color': 'Architecture'},
    title=f"Perplexity: {per} || Coloring by: Architecture (lighter for 'Attn Only' architecture)"
)
fig.show()


layer_labels = sorted(df['layer'].unique(), key=lambda x: int(x))
layer_colors = px.colors.qualitative.Set2[:len(layer_labels)]
layer_color_map = {str(label): color for label, color in zip(layer_labels, layer_colors)}

fig = px.scatter(
    x=projections[:,0], y=projections[:,1],
    color=df['layer'],
    color_discrete_map=layer_color_map,
    labels={'color': 'Layer'},
    title=f"Perplexity: {per} || Coloring by: Layer"
)
fig.show()


point_size = 3  # Set your desired point size

fig = px.scatter(
    x=projections[:,0], y=projections[:,1],
    color=df['arch_label'],
    color_discrete_map=arch_color_full_map,
    labels={'color': 'Architecture'},
    title=f"Perplexity: {per} || Coloring by: Architecture (lighter for 'Attn Only' architecture)",
    size=[point_size]*len(df),  # Set all points to the same size
    size_max=point_size
)

# Set marker border color same as marker color
for trace in fig.data:
    trace.marker.line = dict(width=1, color=trace.marker.color)

fig.show()


fig = px.scatter(
    x=projections[:,0], y=projections[:,1],
    color=df['layer'],
    color_discrete_map=layer_color_map,
    labels={'color': 'Layer'},
    title=f"Perplexity: {per} || Coloring by: Layer",
    size=[point_size]*len(df),
    size_max=point_size
)

for trace in fig.data:
    trace.marker.line = dict(width=1, color=trace.marker.color)

fig.show()


fig = px.scatter(
    x=projections[:,0], y=projections[:,1],
    color=df['l_ratio'],
    color_discrete_map=layer_color_map,
    labels={'color': 'Ratio'},
    title=f"Perplexity: {per} || Coloring by: Where across the model depth the layer containing the head exists? (i.e. Ratio of layer index/total layers in model)",
    size=[point_size]*len(df),
    size_max=point_size
)

for trace in fig.data:
    trace.marker.line = dict(width=1, color=trace.marker.color)

fig.show()

fig = px.scatter(
    x=projections[:,0], y=projections[:,1],
    color=df['num_heads'],
    color_discrete_map={'1': 'rgb(102,194,165)','2': 'rgb(252,141,98)'},
    labels={'color': 'Number of Heads'},
    title=f"Perplexity: {per} || Coloring by: Number of Heads",
    size=[point_size]*len(df),
    size_max=point_size
)

for trace in fig.data:
    trace.marker.line = dict(width=1, color=trace.marker.color)

fig.show()
